# U.S. Medical Insurance Costs

In [ ]:
# Run this cell first if you get ModuleNotFoundError (e.g. sklearn, seaborn)
# Then restart the kernel (Kernel → Restart) and run all cells again
%pip install scikit-learn pandas numpy matplotlib seaborn

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_style('whitegrid')

ModuleNotFoundError: No module named 'sklearn'

In [ ]:
# 1. Load and Inspect
df = pd.read_csv('insurance.csv')

print("Shape:", df.shape)
print("\nDtypes:\n", df.dtypes)
print("\nDescribe:\n", df.describe())
df.info()

# Missing values
print("\nMissing values:\n", df.isnull().sum())


(1338, 7)
age           int64
sex             str
bmi         float64
children      int64
smoker          str
region          str
charges     float64
dtype: object
               age          bmi     children       charges
count  1338.000000  1338.000000  1338.000000   1338.000000
mean     39.207025    30.663397     1.094918  13270.422265
std      14.049960     6.098187     1.205493  12110.011237
min      18.000000    15.960000     0.000000   1121.873900
25%      27.000000    26.296250     0.000000   4740.287150
50%      39.000000    30.400000     1.000000   9382.033000
75%      51.000000    34.693750     2.000000  16639.912515
max      64.000000    53.130000     5.000000  63770.428010
<class 'pandas.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   str    
 2   bmi       1338 non-null   float64
 3   children  

## 2. Univariate Analysis

In [ ]:
# Counts for categorical columns
print("Sex:\n", df['sex'].value_counts())
print("\nSmoker:\n", df['smoker'].value_counts())
print("\nRegion:\n", df['region'].value_counts())
print("\nChildren:\n", df['children'].value_counts())

In [ ]:
# Distributions: histograms for age, bmi, charges
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
df['age'].hist(ax=axes[0], bins=20, edgecolor='black', alpha=0.7)
axes[0].set_title('Age Distribution')
df['bmi'].hist(ax=axes[1], bins=20, edgecolor='black', alpha=0.7)
axes[1].set_title('BMI Distribution')
df['charges'].hist(ax=axes[2], bins=30, edgecolor='black', alpha=0.7)
axes[2].set_title('Charges Distribution')
plt.tight_layout()
plt.show()

In [ ]:
# Box plots for age, bmi, charges
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
df.boxplot(column='age', ax=axes[0])
axes[0].set_title('Age')
df.boxplot(column='bmi', ax=axes[1])
axes[1].set_title('BMI')
df.boxplot(column='charges', ax=axes[2])
axes[2].set_title('Charges')
plt.tight_layout()
plt.show()

## 3. Bivariate Analysis

In [ ]:
# Mean charges by smoker, sex, region
print("Mean charges by smoker:\n", df.groupby('smoker')['charges'].mean())
print("\nMean charges by sex:\n", df.groupby('sex')['charges'].mean())
print("\nMean charges by region:\n", df.groupby('region')['charges'].mean())

## 4. Correlations

In [ ]:
# Charges vs age, bmi, smoker, region
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes[0, 0].scatter(df['age'], df['charges'], alpha=0.5)
axes[0, 0].set_title('Charges vs Age')
axes[0, 1].scatter(df['bmi'], df['charges'], alpha=0.5)
axes[0, 1].set_title('Charges vs BMI')
df.boxplot(column='charges', by='smoker', ax=axes[1, 0])
axes[1, 0].set_title('Charges by Smoker')
df.boxplot(column='charges', by='region', ax=axes[1, 1])
axes[1, 1].set_title('Charges by Region')
plt.suptitle('')
plt.tight_layout()
plt.show()

In [ ]:
# Create copy for modeling
df_model = df.copy()

# Encode categorical variables
df_model['sex'] = LabelEncoder().fit_transform(df_model['sex'])
df_model['smoker'] = LabelEncoder().fit_transform(df_model['smoker'])
df_model = pd.get_dummies(df_model, columns=['region'], prefix='region')

# BMI categories: underweight(<18.5), normal(18.5-25), overweight(25-30), obese(30+)
def bmi_category(bmi):
    if bmi < 18.5: return 0
    elif bmi < 25: return 1
    elif bmi < 30: return 2
    else: return 3

df_model['bmi_category'] = df_model['bmi'].apply(bmi_category)
df_model['smoker_bmi'] = df_model['smoker'] * df_model['bmi']

print(df_model.head())

In [ ]:
# Correlation matrix and heatmap
corr = df.select_dtypes(include=[np.number]).corr()
print("Correlation matrix:\n", corr)

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()

## 5. Feature Engineering

## 6. Train-Test Split

In [ ]:
# Define features and target
X = df_model.drop('charges', axis=1)
y = df_model['charges']

# 80/20 split with random_state for reproducibility
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features for linear models
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")

In [ ]:
# Detect outliers using IQR for charges and bmi
def remove_outliers_iqr(data, columns):
    df_clean = data.copy()
    for col in columns:
        Q1 = df_clean[col].quantile(0.25)
        Q3 = df_clean[col].quantile(0.75)
        IQR = Q3 - Q1
        df_clean = df_clean[(df_clean[col] >= Q1 - 1.5*IQR) & (df_clean[col] <= Q3 + 1.5*IQR)]
    return df_clean

df_no_outliers = remove_outliers_iqr(df, ['charges', 'bmi'])
print(f"Original: {len(df)} rows | After removing outliers: {len(df_no_outliers)} rows")

In [ ]:
def evaluate_model(model, X_tr, X_te, y_tr, y_te, name):
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    return {
        'Model': name,
        'MAE': mean_absolute_error(y_te, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_te, y_pred)),
        'R2': r2_score(y_te, y_pred)
    }

# Use unscaled for tree models, scaled for linear
results = []

# Linear Regression (baseline)
results.append(evaluate_model(LinearRegression(), X_train_scaled, X_test_scaled, y_train, y_test, 'Linear Regression'))

# Ridge
results.append(evaluate_model(Ridge(alpha=1.0), X_train_scaled, X_test_scaled, y_train, y_test, 'Ridge'))

# Lasso
results.append(evaluate_model(Lasso(alpha=1.0), X_train_scaled, X_test_scaled, y_train, y_test, 'Lasso'))

# Random Forest (unscaled)
results.append(evaluate_model(RandomForestRegressor(random_state=42), X_train, X_test, y_train, y_test, 'Random Forest'))

# Gradient Boosting (unscaled)
results.append(evaluate_model(GradientBoostingRegressor(random_state=42), X_train, X_test, y_train, y_test, 'Gradient Boosting'))

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

In [ ]:
# Linear model coefficients (use scaled features)
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
coef_df = pd.DataFrame({'feature': X.columns, 'coefficient': lr.coef_})
coef_df['abs_coef'] = np.abs(coef_df['coefficient'])
coef_df = coef_df.sort_values('abs_coef', ascending=False)
print("Linear Regression Coefficients (which features drive charges most):\n", coef_df)

# Random Forest feature importances
rf = RandomForestRegressor(random_state=42)
rf.fit(X_train, y_train)
imp_df = pd.DataFrame({'feature': X.columns, 'importance': rf.feature_importances_}).sort_values('importance', ascending=False)
print("\nRandom Forest Feature Importances:\n", imp_df)

# Plot feature importances
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
coef_df.plot(x='feature', y='coefficient', kind='barh', ax=axes[0], legend=False)
axes[0].set_title('Linear Regression Coefficients')
imp_df.plot(x='feature', y='importance', kind='barh', ax=axes[1], legend=False)
axes[1].set_title('Random Forest Feature Importances')
plt.tight_layout()
plt.show()

## 9. Outlier Handling

In [ ]:
# Re-engineer features and compare model performance with/without outliers
df_clean_model = df_no_outliers.copy()
df_clean_model['sex'] = LabelEncoder().fit_transform(df_clean_model['sex'])
df_clean_model['smoker'] = LabelEncoder().fit_transform(df_clean_model['smoker'])
df_clean_model = pd.get_dummies(df_clean_model, columns=['region'], prefix='region')
df_clean_model['bmi_category'] = df_clean_model['bmi'].apply(bmi_category)
df_clean_model['smoker_bmi'] = df_clean_model['smoker'] * df_clean_model['bmi']

X_clean = df_clean_model.drop('charges', axis=1)
y_clean = df_clean_model['charges']
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_clean, y_clean, test_size=0.2, random_state=42)
scaler_c = StandardScaler()
X_train_cs = scaler_c.fit_transform(X_train_c)
X_test_cs = scaler_c.transform(X_test_c)

# Compare RF on full vs cleaned data
rf_full = evaluate_model(RandomForestRegressor(random_state=42), X_train, X_test, y_train, y_test, 'RF (with outliers)')
rf_clean = evaluate_model(RandomForestRegressor(random_state=42), X_train_c, X_test_c, y_train_c, y_test_c, 'RF (no outliers)')
print(pd.DataFrame([rf_full, rf_clean]).to_string(index=False))

## 10. Model Tuning

## 12. Business Insights

In [ ]:
# Grid search for Random Forest
param_grid = {'n_estimators': [50, 100, 200], 'max_depth': [5, 10, 15, None], 'min_samples_split': [2, 5]}
grid_rf = GridSearchCV(RandomForestRegressor(random_state=42), param_grid, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
grid_rf.fit(X_train, y_train)
print("Best RF params:", grid_rf.best_params_)
print("Best CV RMSE:", np.sqrt(-grid_rf.best_score_))

# 5-fold cross-validation for top models
models = [('Linear', LinearRegression()), ('Ridge', Ridge(alpha=1.0)), ('RF', RandomForestRegressor(random_state=42))]
for name, model in models:
    X_tr = X_train_scaled if name != 'RF' else X_train
    X_te = X_test_scaled if name != 'RF' else X_test
    scores = cross_val_score(model, X_tr, y_train, cv=5, scoring='r2')
    print(f"{name} CV R²: {scores.mean():.3f} (+/- {scores.std()*2:.3f})")

In [ ]:
# Fit best linear model and analyze residuals
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
y_pred = lr.predict(X_test_scaled)
residuals = y_test - y_pred

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Residuals vs predicted
axes[0].scatter(y_pred, residuals, alpha=0.5)
axes[0].axhline(y=0, color='r', linestyle='--')
axes[0].set_xlabel('Predicted charges')
axes[0].set_ylabel('Residuals')
axes[0].set_title('Residuals vs Predicted')

# Normality of residuals (histogram)
axes[1].hist(residuals, bins=30, edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Residual')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Residuals (normality check)')
plt.tight_layout()
plt.show()

In [ ]:
# Summarize how age, BMI, smoking, region affect charges
print("=== KEY FINDINGS ===\n")

smoker_means = df.groupby('smoker')['charges'].mean()
print(f"1. SMOKING: Smokers pay ~${smoker_means['yes'] - smoker_means['no']:,.0f} more on average")
print(f"   Non-smokers: ${smoker_means['no']:,.0f} | Smokers: ${smoker_means['yes']:,.0f}\n")

# Age effect (compare young vs old)
young = df[df['age'] < 35]['charges'].mean()
old = df[df['age'] >= 50]['charges'].mean()
print(f"2. AGE: Older enrollees (50+) pay ~${old - young:,.0f} more than younger (<35)")
print(f"   Young: ${young:,.0f} | Older: ${old:,.0f}\n")

# BMI effect (obese vs normal: BMI 30+ vs 18.5-25)
normal_bmi = df[(df['bmi'] >= 18.5) & (df['bmi'] < 25)]['charges'].mean()
obese_bmi = df[df['bmi'] >= 30]['charges'].mean()
print(f"3. BMI: Obese (BMI≥30) pay ~${obese_bmi - normal_bmi:,.0f} more than normal weight")
print(f"   Normal BMI: ${normal_bmi:,.0f} | Obese: ${obese_bmi:,.0f}\n")

print("4. REGION: Southeast has highest mean charges; regional differences are smaller than smoking/age/BMI")

print("\n=== ACTIONABLE INSIGHTS FOR INSURERS ===")
print("• Target smoking cessation programs — largest cost driver")
print("• Consider age-banded pricing — clear correlation with charges")
print("• Wellness/weight programs may reduce claims for high-BMI enrollees")